# Kidney Histopathology Classification

Implement the requested functions in order. Setup, data loading and reporting utilities are supplied. Cells raising `NotImplementedError` are intentional exercises: complete them before running their checks or dependent sections. Written explanations belong in the answer cells.


### Overview
The challenge is to classify a kidney histopathology patch into **Chromo, Onco, ccRCC, normal or pRCC**. We use the supplied training, validation and test splits. The dataset uses balanced train/validation sets and a different class distribution in test.

| Strategy in the challenge | Implementation in this lab |
|---|---|
| Colour features | Normalised RGB histograms |
| Texture features | Uniform local binary patterns and grey-level co-occurrence statistics |
| Morphological features | Thresholded dark-region area and shape summaries |
| Classical classification | StandardScaler and multinomial logistic regression |
| CNN trained from scratch | Small three-block convolutional network |
| Transfer learning | ResNet18 head training then last-block fine-tuning |
| Additional pretrained architectures | Complete EfficientNet-B0 and ViT-B/16 adapters and training paths |
| Hybrid approach | Concatenate handcrafted features with frozen ResNet18 embeddings; compare with deep features alone |



**Data setup:** place the downloaded competition ZIP in a `data/` folder beside this notebook, or extract it there so that `data/Histo images/train`, `val` and `test` exist.



In [ ]:
# Uncomment in a dedicated environment if dependencies are missing.
# %pip install torch torchvision numpy pandas pillow matplotlib scikit-learn scikit-image ipython joblib
# On a local GPU, install a compatible torch/torchvision pair using pytorch.org first.


In [ ]:
from pathlib import Path
import os, sys, json, copy, random, time, hashlib, zipfile, platform, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from IPython.display import display
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from skimage.color import rgb2gray
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.morphology import opening, disk
from skimage.measure import label, regionprops
import joblib

CLASSES = ['Chromo', 'Onco', 'ccRCC', 'normal', 'pRCC']
CLASS_TO_IDX = {name:i for i,name in enumerate(CLASSES)}
PROJECT_DIR = Path.cwd().resolve()
DATA_ROOT = ''  # Set only for an already extracted external dataset; must contain train/val.
DATA_ZIP = ''   # Optional path to the already downloaded ZIP, e.g. on Drive.
OUTPUT_ROOT = PROJECT_DIR/'outputs'  # Change to a mounted Drive path to persist Colab results.
SEED = 42
IMAGE_SIZE = 224
FEATURE_SIZE = 128
BATCH_SIZE = 16  # 8 or 4 for limited GPU memory; ViT may need a smaller batch.
CNN_EPOCHS = 5
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 2
MAX_TRAIN_PER_CLASS = 0  # 0=all; same selected training rows for every method.
PRETRAINED = True
RUN_EXTRA_ARCHITECTURES = False  # Set True to train EfficientNet-B0 and ViT-B/16 too.
RUN_FINAL_TEST = False          # Activate only after model selection is finished.

# Software smoke mode is for validation only: random images/weights, no scientific results.
SMOKE = os.environ.get('KIDNEY_LAB_SMOKE') == '1'
if SMOKE:
    DATA_ROOT = os.environ['KIDNEY_LAB_TEST_DATA']
    OUTPUT_ROOT = Path(os.environ.get('KIDNEY_LAB_TEST_OUTPUT', '/tmp/kidney_lab_checks'))
    IMAGE_SIZE = 64; FEATURE_SIZE = 64; BATCH_SIZE = 5
    CNN_EPOCHS = HEAD_EPOCHS = FINETUNE_EPOCHS = 1
    PRETRAINED = False
    print('SYNTHETIC SOFTWARE CHECK: random weights; not a histology experiment.')

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
torch.set_num_threads(2)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available() else 'cpu')
OUT = OUTPUT_ROOT/('run_'+time.strftime('%Y%m%d_%H%M%S')+'_'+str(time.time_ns())[-6:])
OUT.mkdir(parents=True, exist_ok=False)
VERSIONS = {'python':platform.python_version(), 'torch':str(torch.__version__),
            'torchvision':str(torchvision.__version__), 'device':str(DEVICE)}
print(VERSIONS); print('Outputs:', OUT)


In [ ]:
# Supplied infrastructure: local ZIP preparation and explicit class mapping.
EXTENSIONS = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def locate_roots(folder):
    folder=Path(folder)
    roots=set()
    if (folder/'train').is_dir() and (folder/'val').is_dir(): roots.add(folder.resolve())
    if folder.is_dir():
        for p in folder.rglob('train'):
            if p.is_dir() and (p.parent/'val').is_dir(): roots.add(p.parent.resolve())
    return sorted(roots)

def prepare_dataset():
    if DATA_ROOT:
        root=Path(DATA_ROOT).expanduser()
        if not root.is_absolute(): root=PROJECT_DIR/root
        if not (root/'train').is_dir() or not (root/'val').is_dir():
            raise FileNotFoundError(f'DATA_ROOT must contain train/ and val/: {root}')
        return root.resolve()
    directory=PROJECT_DIR/'data'; directory.mkdir(exist_ok=True)
    roots=locate_roots(directory)
    if not roots and Path('/kaggle/input').is_dir(): roots=locate_roots('/kaggle/input')
    if len(roots)>1: raise ValueError('Multiple datasets found. Set DATA_ROOT explicitly.')
    if roots: return roots[0]
    if DATA_ZIP:
        archive=Path(DATA_ZIP).expanduser()
        if not archive.is_absolute(): archive=PROJECT_DIR/archive
    else:
        archives=sorted(p for p in directory.iterdir() if p.suffix.lower()=='.zip')
        if len(archives)!=1:
            raise FileNotFoundError('Put one dataset ZIP into data/, or set DATA_ZIP / DATA_ROOT.')
        archive=archives[0]
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target=(directory/member.filename.replace('\\','/')).resolve()
            if not target.is_relative_to(directory.resolve()): raise ValueError('Archive path escapes data/')
            if (member.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('Symlinks unsupported')
        print('Extracting dataset; this may take several minutes.')
        z.extractall(directory)
    roots=locate_roots(directory)
    if len(roots)!=1: raise ValueError('Could not find a unique train/val root after extraction.')
    return roots[0]

def scan_split(root, split):
    folder=Path(root)/split
    if not folder.is_dir(): raise FileNotFoundError(folder)
    unknown=[p.name for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.') and p.name not in CLASSES]
    if unknown: raise ValueError(f'Unexpected class folders: {unknown}')
    rows=[]
    for cls in CLASSES:
        paths=sorted(p for p in (folder/cls).rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
        if not paths: raise ValueError(f'Empty/missing class folder: {folder/cls}')
        rows.extend({'path':str(p.resolve()),'relative_path':p.relative_to(root).as_posix(),
                     'class_name':cls,'label':CLASS_TO_IDX[cls],'split':split} for p in paths)
    return pd.DataFrame(rows)

ROOT=prepare_dataset()
train_full=scan_split(ROOT,'train'); val=scan_split(ROOT,'val')
train=train_full.copy()
if MAX_TRAIN_PER_CLASS:
    if MAX_TRAIN_PER_CLASS > int(train_full.groupby('label').size().min()):
        raise ValueError('MAX_TRAIN_PER_CLASS exceeds the smallest training class.')
    train=train_full.groupby('label',group_keys=False).sample(n=MAX_TRAIN_PER_CLASS,random_state=SEED)
train=train.reset_index(drop=True); val=val.reset_index(drop=True)
train.to_csv(OUT/'training_rows.csv',index=False); val.to_csv(OUT/'validation_rows.csv',index=False)
print('Data root:', ROOT, '\nClass mapping:', CLASS_TO_IDX)
# Test is intentionally not read until the final-test section.


## Task 1 — Explore the dataset


Compute a class-count table for train and validation. Display three randomly selected training patches per class using a fixed seed and show their original pixel dimensions. Explain how class labels differ from segmentation masks. Verify the dataset's balance claim rather than assuming it. Record that comparisons use a subset if MAX_TRAIN_PER_CLASS is nonzero.

**Contract:** `explore_data(train, val, seed)` returns a DataFrame indexed by CLASSES with columns train/val, and a matplotlib Figure with five rows. Do not read test.


In [ ]:
def explore_data(train, val, seed=SEED):
    # TODO 1: count each class in both frames; retain the CLASSES row order.
    # TODO 2: sample up to three images per class without replacement using seed.
    # TODO 3: display RGB images, class names and original dimensions in a 5x3 grid.
    # Return counts, fig.
    raise NotImplementedError('Complete Task 1')


In [ ]:
counts,fig=explore_data(train,val)
assert counts.shape==(5,2)
assert counts['train'].sum()==len(train) and counts['val'].sum()==len(val)
display(counts); plt.show()


**Your explanation:** Write your reasoning and interpret the results you obtain.


## Task 2 — Audit leakage and data integrity


Read each image to confirm it decodes. Compute a SHA256 digest of its file bytes. Compare the **full** training split with validation even if a training subset is used later. Detect cross-split overlap and stop before fitting if duplicates exist. Explain what exact hashing misses.

**Contract:** `audit_frames(frames)` returns a DataFrame retaining the input columns plus sha256, width, height and mode. `find_cross_split_duplicates(audit)` returns all rows whose digest appears in more than one split. Use supplied splits; do not invent patient IDs.


In [ ]:
def audit_frames(frames):
    # TODO: concatenate input frames, stream file bytes into SHA256, open each image.
    # Preserve each row and append sha256, width, height and mode. Fail on unreadable files.
    raise NotImplementedError('Complete Task 2 audit')

def find_cross_split_duplicates(audit):
    # TODO: select digests present in more than one distinct split and return all their rows.
    raise NotImplementedError('Complete Task 2 overlap check')


In [ ]:
audit=audit_frames([train_full,val])
duplicates=find_cross_split_duplicates(audit)
audit.to_csv(OUT/'train_val_audit.csv',index=False)
duplicates.to_csv(OUT/'cross_split_duplicates.csv',index=False)
assert len(audit)==len(train_full)+len(val)
if not duplicates.empty:
    raise ValueError('Cross-split duplicates found. Review the report and resolve the protocol before training.')
display(audit.groupby('split')[['width','height']].agg(['min','median','max']))


**Your explanation:** Write your reasoning and interpret the results you obtain.


### Shared feature input
Handcrafted features use a deterministic 128-pixel view of the entire image. Aspect ratio is preserved and white padding is used. Pixel-area measurements therefore refer to this analysis grid, not calibrated physical units. The coarse threshold-based regions below are **morphological proxies**, not validated nuclei or renal structures: no stain identity or segmentation annotations are established here.


In [ ]:
def read_feature_rgb(path):
    with Image.open(path) as im:
        im=ImageOps.pad(im.convert('RGB'),(FEATURE_SIZE,FEATURE_SIZE),
                        method=Image.Resampling.BILINEAR,color=(255,255,255))
        return np.asarray(im,dtype=np.uint8)

HIST_BINS=16
FEATURE_DIMS={'colour':3*HIST_BINS,'texture':14,'morphology':8}


## Task 3 — Extract colour histograms


Implement one histogram for each RGB channel with HIST_BINS bins over [0, 256]. Normalise each channel histogram to sum to one, then concatenate them. Use no labels during extraction. What spatial information is lost?

**Contract:** input H×W×3 uint8; output float32 vector of length `3*bins`.


In [ ]:
def colour_features(rgb,bins=HIST_BINS):
    # TODO: normalised histogram for each channel, concatenated in R,G,B order.
    raise NotImplementedError('Complete Task 3')


In [ ]:
rgb=read_feature_rgb(train.iloc[0].path)
c=colour_features(rgb)
assert c.shape==(3*HIST_BINS,) and np.isfinite(c).all()
np.testing.assert_allclose(c.reshape(3,HIST_BINS).sum(1),1,atol=1e-6)


**Your explanation:** Write your reasoning and interpret the results you obtain.


## Task 4 — Describe texture with LBP and GLCM


Convert RGB to uint8 greyscale. Compute uniform LBP with P=8 and R=1, and normalise its 10-bin histogram. Quantise greyscale into 16 levels for a symmetric normalised GLCM using distances [1,2] and angles [0, π/4, π/2, 3π/4]. Append the mean contrast, homogeneity, energy and correlation across distances/angles.

**Contract:** return 14 finite float32 values. Use integer greyscale for LBP to avoid floating-point threshold instability. Explain why averaging directions reduces, but does not guarantee, rotation invariance.


In [ ]:
def texture_features(rgb):
    # TODO: uint8 greyscale -> uniform LBP -> normalised 10-bin histogram.
    # TODO: 16-level GLCM with the specified distances/angles -> four averaged statistics.
    # Return a finite float32 vector of length 14, handling degenerate images.
    raise NotImplementedError('Complete Task 4')


In [ ]:
t=texture_features(rgb)
assert t.shape==(14,) and np.isfinite(t).all()
np.testing.assert_allclose(t[:10].sum(),1,atol=1e-6)
assert np.isfinite(texture_features(np.full((32,32,3),255,dtype=np.uint8))).all()


**Your explanation:** Write your reasoning and interpret the results you obtain.


## Task 5 — Build morphological proxies


Threshold dark regions with greyscale < Otsu threshold; use a radius-1 binary opening. Retain connected components with area between 8 pixels and 25% of the analysis image. Handle constant/empty images explicitly. Calculate:
1. retained foreground fraction; 2. component count per 1000 pixels;
3–4. mean/std component area divided by image area;
5. mean eccentricity; 6. mean solidity; 7. mean extent;
8. median circularity `4π area / perimeter²`, clipped to [0,1].

**Contract:** `morphology_features(rgb, return_mask=False)` returns eight finite floats; when return_mask=True, return `(features, retained_boolean_mask)`. Visualise the mask and explain why it must not be called a nucleus segmentation.


In [ ]:
def morphology_features(rgb,return_mask=False):
    # TODO: threshold/open/label; filter components by the stated area limits.
    # TODO: compute eight summaries, returning zeros when no components remain.
    # TODO: optionally return the retained boolean mask for inspection.
    raise NotImplementedError('Complete Task 5')


In [ ]:
m,mask=morphology_features(rgb,return_mask=True)
assert m.shape==(8,) and mask.shape==rgb.shape[:2] and np.isfinite(m).all()
assert np.all(morphology_features(np.full((32,32,3),255,dtype=np.uint8))==0)
fig,ax=plt.subplots(1,2,figsize=(6,3))
ax[0].imshow(rgb);ax[0].set_title('Input');ax[1].imshow(mask,cmap='gray');ax[1].set_title('Dark-region proxy')
for a in ax:a.axis('off')
plt.show()


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
# Supplied extraction loop keeps feature rows aligned with each DataFrame.
def extract_handcrafted(frame):
    blocks={key:[] for key in FEATURE_DIMS}
    for path in frame.path:
        image=read_feature_rgb(path)
        blocks['colour'].append(colour_features(image))
        blocks['texture'].append(texture_features(image))
        blocks['morphology'].append(morphology_features(image))
    result={key:np.stack(values) for key,values in blocks.items()}
    for key,X in result.items():
        assert X.shape==(len(frame),FEATURE_DIMS[key]) and np.isfinite(X).all()
    return result

start=time.perf_counter()
hand_train=extract_handcrafted(train)
hand_val=extract_handcrafted(val)
handcrafted_seconds=time.perf_counter()-start
H_train=np.concatenate(list(hand_train.values()),axis=1)
H_val=np.concatenate(list(hand_val.values()),axis=1)
np.savez_compressed(OUT/'handcrafted_features.npz',train=H_train,val=H_val)
y_train=train.label.to_numpy(); y_val=val.label.to_numpy()


## Task 6 — Evaluate and fit classical baselines


Implement accuracy, balanced accuracy and macro F1, then fit StandardScaler + LogisticRegression on **training rows only**. Use max_iter=2000, C=1 and the shared seed. Return the fitted pipeline. Compare the constant reference, individual descriptor families and their concatenation using the supplied experiment loop.

**Contract:** `scores(y,pred)` returns keys accuracy/balanced_accuracy/macro_f1. `fit_linear(X,y)` returns a fitted pipeline with predict_proba. Macro F1 must include the five fixed labels, with zero_division=0.


In [ ]:
def scores(y,pred):
    # TODO: compute the three metrics with the fixed five-class convention.
    raise NotImplementedError('Complete Task 6 metrics')

def fit_linear(X,y):
    # TODO: build and fit the preprocessing/classifier pipeline on training data only.
    raise NotImplementedError('Complete Task 6 classifier')


In [ ]:
toy_y=np.arange(5);toy_pred=np.zeros(5,dtype=int)
assert abs(scores(toy_y,toy_pred)['balanced_accuracy']-.2)<1e-9


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
# Supplied experiment registry. Validation probabilities are always in CLASS_TO_IDX order.
RESULTS=[]; ARTIFACTS={}; VAL_PROBS={}

def ordered_probabilities(estimator,X):
    p=estimator.predict_proba(X)
    if set(estimator.classes_)!=set(range(5)): raise ValueError('Classifier is missing classes.')
    return p[:,[list(estimator.classes_).index(i) for i in range(5)]]

def record(name,prob,artifact,seconds=0.):
    if prob.shape!=(len(val),5): raise ValueError('Validation probability dimensions do not match.')
    if not np.isfinite(prob).all(): raise ValueError('Nonfinite predictions.')
    np.testing.assert_allclose(prob.sum(1),1,atol=1e-5)
    RESULTS.append({'method':name,**scores(y_val,prob.argmax(1)),'fit_seconds':seconds})
    ARTIFACTS[name]=artifact; VAL_PROBS[name]=prob
    pd.DataFrame(RESULTS).to_csv(OUT/'validation_comparison.csv',index=False)

majority=int(np.bincount(y_train,minlength=5).argmax())
record('constant',np.eye(5)[np.full(len(val),majority)],{'kind':'constant','label':majority})
for key in ['colour','texture','morphology','all']:
    start=time.perf_counter()
    A,B=(H_train,H_val) if key=='all' else (hand_train[key],hand_val[key])
    linear=fit_linear(A,y_train)
    record('classical_'+key,ordered_probabilities(linear,B),
           {'kind':'classical','feature':key,'estimator':linear},time.perf_counter()-start)
    joblib.dump(linear,OUT/f'classical_{key}.joblib')
display(pd.DataFrame(RESULTS))


### Neural models: data and input conventions
Use the same image rows as the classical experiments. Training may use random flips and modest colour jitter; validation/test preprocessing is deterministic. All neural inputs are RGB, aspect-ratio-preserving padded images followed by ImageNet channel normalisation. This is one explicit lab preprocessing policy, not a claim that it exactly reproduces every weight enum's default transform. Use 224 pixels for ViT-B/16. Strong augmentation or changing tissue geometry requires justification.


In [ ]:
# Supplied portable Dataset and padding operation; num_workers=0 also works on Windows.
class PadResize:
    def __init__(self,size): self.size=size
    def __call__(self,im):
        return ImageOps.pad(im,(self.size,self.size),method=Image.Resampling.BILINEAR,
                            color=(255,255,255))

class PatchDataset(Dataset):
    def __init__(self,frame,transform):
        self.frame=frame.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with Image.open(row.path) as im: image=im.convert('RGB')
        return self.transform(image),int(row.label)

def make_loader(frame,transform,shuffle=False):
    generator=torch.Generator().manual_seed(SEED)
    return DataLoader(PatchDataset(frame,transform),batch_size=BATCH_SIZE,shuffle=shuffle,
                      num_workers=0,generator=generator,pin_memory=DEVICE.type=='cuda')


## Task 7 — Build augmentation and a CNN from scratch


Create training and evaluation transforms. Use random horizontal/vertical flips and modest colour jitter only for training. Convert to tensors and normalise using ImageNet means (0.485, 0.456, 0.406) and standard deviations (0.229, 0.224, 0.225).

Implement a small CNN with three convolution/ReLU/pooling blocks, adaptive global pooling and a five-output linear head. Return **raw logits**, because cross-entropy includes the necessary log-softmax. This network is randomly initialised and trained end to end.

**Contracts:** `make_transforms(size)` returns `(train_transform, eval_transform)`. `SmallCNN(num_classes=5)` accepts batches shaped (N,3,H,W) and returns (N,5). Explain why model capacity and augmentation matter when data are limited.


In [ ]:
def make_transforms(size=IMAGE_SIZE):
    # TODO: build separate training and deterministic evaluation pipelines.
    # Use the supplied PadResize and torchvision.transforms operations.
    raise NotImplementedError('Complete Task 7 transforms')

class SmallCNN(nn.Module):
    def __init__(self,num_classes=5):
        super().__init__()
        # TODO: define three convolutional blocks, global pooling and a linear head.
        raise NotImplementedError('Complete Task 7 CNN layers')
    def forward(self,x):
        # TODO: extract features, flatten pooled features, return logits.
        raise NotImplementedError('Complete Task 7 CNN forward')


In [ ]:
train_transform,eval_transform=make_transforms()
train_loader=make_loader(train,train_transform,shuffle=True)
val_loader=make_loader(val,eval_transform)
xb,yb=next(iter(train_loader))
assert xb.shape[1:]==(3,IMAGE_SIZE,IMAGE_SIZE) and yb.min()>=0 and yb.max()<5
with Image.open(train.iloc[0].path) as im:
    a=eval_transform(im.convert('RGB')); b=eval_transform(im.convert('RGB'))
assert torch.equal(a,b), 'Validation transforms must be deterministic.'
with torch.no_grad(): assert SmallCNN()(xb[:2]).shape==(2,5)


**Your explanation:** Write your reasoning and interpret the results you obtain.


## Task 8 — Implement and verify a training step


Implement one supervised optimisation step: clear gradients, compute logits and cross-entropy, backpropagate, and update trainable parameters. Return the detached loss and number of correct predictions. The supplied loop below handles device transfer, modes, validation and checkpoint selection; study it carefully.

**Contract:** `training_step(model, images, targets, optimizer, criterion)` returns `(float_loss, int_correct)`. Inputs already reside on DEVICE. Explain `train()` vs `eval()` and why validation does not update weights.


In [ ]:
def training_step(model,images,targets,optimizer,criterion):
    # TODO: reset gradients, forward pass, loss, backward pass and optimizer update.
    # Return a Python float loss and an integer count; do not retain the graph.
    raise NotImplementedError('Complete Task 8')


In [ ]:
probe=nn.Linear(3,5).to(DEVICE)
probe_opt=torch.optim.SGD(probe.parameters(),lr=.1)
before=probe.weight.detach().clone()
loss,n_correct=training_step(probe,torch.ones(4,3,device=DEVICE),
    torch.tensor([0,1,2,3],device=DEVICE),probe_opt,nn.CrossEntropyLoss())
assert np.isfinite(loss) and 0<=n_correct<=4 and not torch.equal(before,probe.weight)
del probe,probe_opt


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
# Supplied loops. Selection metric is validation macro F1 throughout all stages.
@torch.inference_mode()
def predict_neural(model,loader):
    model.eval(); batches=[]
    for images,_ in loader:
        batches.append(model(images.to(DEVICE)).softmax(1).cpu().numpy())
    return np.concatenate(batches)

def load_checkpoint(path):
    ckpt=torch.load(path,map_location='cpu',weights_only=True)
    if ckpt['classes']!=CLASSES: raise ValueError('Checkpoint class mapping mismatch.')
    return ckpt

def fit_stage(model,name,arch,epochs,lr,freeze_bn=False,best_score=-1.,stage='train'):
    model.to(DEVICE)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=lr,weight_decay=1e-4)
    criterion=nn.CrossEntropyLoss(); history=[]
    checkpoint=OUT/f'{name}.pt'
    for epoch in range(epochs):
        model.train()
        if freeze_bn:
            # Keep pretrained BatchNorm running statistics fixed, including in the last block.
            for module in model.modules():
                if isinstance(module,nn.modules.batchnorm._BatchNorm): module.eval()
        total_loss=0.; total_correct=0; total=0
        for images,targets in train_loader:
            images=images.to(DEVICE); targets=targets.to(DEVICE)
            loss,correct=training_step(model,images,targets,optimizer,criterion)
            total_loss+=loss*len(targets); total_correct+=correct; total+=len(targets)
        prob=predict_neural(model,val_loader)
        metric=scores(y_val,prob.argmax(1))
        row={'stage':stage,'epoch':epoch+1,'train_loss':total_loss/total,
             'train_accuracy':total_correct/total,**{'val_'+k:v for k,v in metric.items()}}
        history.append(row); print(name,row)
        if metric['macro_f1']>best_score:
            best_score=metric['macro_f1']
            state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save({'state_dict':state,'classes':CLASSES,'arch':arch,'image_size':IMAGE_SIZE,
                        'pretrained_initialisation':bool(PRETRAINED and arch!='small_cnn'),
                        'best_val_macro_f1':best_score,'stage':stage,'seed':SEED},checkpoint)
    return best_score,history,checkpoint

def finish_neural(model,name,arch,checkpoint,history,seconds):
    model.load_state_dict(load_checkpoint(checkpoint)['state_dict']); model.to(DEVICE)
    prob=predict_neural(model,val_loader)
    record(name,prob,{'kind':'neural','arch':arch,'checkpoint':str(checkpoint)},seconds)
    pd.DataFrame(history).to_csv(OUT/f'{name}_history.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(9,3))
    axes[0].plot(range(1,len(history)+1),[r['train_loss'] for r in history],marker='o')
    axes[0].set(title=name+' training loss',xlabel='Epoch',ylabel='Cross-entropy')
    axes[1].plot(range(1,len(history)+1),[r['val_macro_f1'] for r in history],marker='o')
    axes[1].set(title='Validation macro F1',xlabel='Epoch',ylabel='Macro F1',ylim=(0,1))
    fig.tight_layout();fig.savefig(OUT/f'{name}_learning_curves.png',dpi=140);plt.show()
    model.cpu()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

set_seed(); cnn=SmallCNN()
start=time.perf_counter()
_,history,checkpoint=fit_stage(cnn,'cnn_scratch','small_cnn',CNN_EPOCHS,1e-3)
finish_neural(cnn,'cnn_scratch','small_cnn',checkpoint,history,time.perf_counter()-start)
del cnn;gc.collect()


## Task 9 — Compare ImageNet and histology pretraining, then fine-tune the last block

Implement transfer learning using two sources of pretrained weights:

- **ImageNet pretraining:** implement architecture adapters for **ResNet18, EfficientNet-B0 and ViT-B/16**. Load their `DEFAULT` ImageNet weights when `pretrained=True` and replace the classification layer with a new five-output linear layer.
- **Histology pretraining:** load the **Ciga et al. histology-pretrained ResNet18** checkpoint. Remove its self-supervised projection head, verify that all backbone parameters are loaded correctly, and attach a new five-output linear classifier. Use the separate checkpoint-loading cell provided for this experiment.

Do not swallow download errors or silently substitute random weights. Record the histology checkpoint’s source and checksum.

Implement two training policies:

1. **Head training:** `head` trains only the new classifier while the backbone remains frozen.
2. **Last-block fine-tuning:** `last_block` trains the classifier and the final feature block using a smaller learning rate. All other parameters remain frozen.

For ResNet18, fine-tune `layer4`; for EfficientNet-B0, use `features[-1]`; for ViT-B/16, use the last transformer encoder layer and keep the final encoder normalisation frozen. Keep BatchNorm running statistics fixed during both transfer stages.

Start fine-tuning from the best head-training checkpoint. Retain the best checkpoint across both stages according to **validation macro F1**.

Compare ImageNet-pretrained and histology-pretrained ResNet18 using the same training/validation splits, preprocessing, augmentation and epoch budget. Report **accuracy, balanced accuracy and macro F1**, and inspect their **validation confusion matrices**. Keep the test set untouched during this comparison. EfficientNet-B0 and ViT-B/16 runs are optional extensions.

**Contracts:**

- `build_transfer(arch, pretrained=True)` returns `(model, head, last_block)` for the ImageNet adapters. Valid architecture names remain `resnet18`, `efficientnet_b0` and `vit_b_16`.
- `configure_transfer(model, head, last_block, stage)` changes `requires_grad` in place.
- The histology experiment uses a separate loading cell and is registered as `transfer_resnet18_histology`, while retaining `arch="resnet18"` for checkpoint reloading.

**Questions:** Explain transfer learning versus training from scratch, and head training versus fine-tuning. Does histology pretraining improve performance on this dataset? Support your answer with the measured results and class-specific errors.

In [ ]:
def build_transfer(arch,pretrained=True):
    # TODO: use the appropriate torchvision constructor and DEFAULT weights enum.
    # ResNet: model.fc, model.layer4
    # EfficientNet: model.classifier[-1], model.features[-1]
    # ViT: model.heads.head, model.encoder.layers[-1]
    # Replace only the classification layer. Return model, head module, last block.
    raise NotImplementedError('Complete Task 9 architecture adapters')

def configure_transfer(model,head,last_block,stage):
    # TODO: first freeze everything, then enable head gradients.
    # For stage='last_block', also enable gradients for that block.
    # Reject unknown stage names.
    raise NotImplementedError('Complete Task 9 freeze policy')


In [ ]:
# A lightweight structural check; this random-weight model is never an experiment result.
probe,head,block=build_transfer('resnet18',pretrained=False)
configure_transfer(probe,head,block,'head')
assert all(p.requires_grad for p in head.parameters())
assert not any(p.requires_grad for p in block.parameters())
configure_transfer(probe,head,block,'last_block')
assert all(p.requires_grad for p in block.parameters())
assert not any(p.requires_grad for p in probe.layer1.parameters())
del probe,head,block;gc.collect()


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
def train_transfer(arch):
    if arch=='vit_b_16' and IMAGE_SIZE!=224:
        raise ValueError('ViT-B/16 in this lab requires IMAGE_SIZE=224. Re-run preprocessing first.')
    set_seed(); model,head,block=build_transfer(arch,PRETRAINED)
    name='transfer_'+arch
    start=time.perf_counter()
    configure_transfer(model,head,block,'head')
    best,first,checkpoint=fit_stage(model,name,arch,HEAD_EPOCHS,1e-3,True,stage='head')
    model.load_state_dict(load_checkpoint(checkpoint)['state_dict'])
    configure_transfer(model,head,block,'last_block')
    _,second,checkpoint=fit_stage(model,name,arch,FINETUNE_EPOCHS,1e-5,True,
                                  best_score=best,stage='last_block')
    # Both stages compete for the same checkpoint; a worse fine-tuned model is not retained.
    finish_neural(model,name,arch,checkpoint,first+second,time.perf_counter()-start)
    del model,head,block;gc.collect()

train_transfer('resnet18')
if RUN_EXTRA_ARCHITECTURES:
    for arch in ['efficientnet_b0','vit_b_16']: train_transfer(arch)
else:
    print('EfficientNet/ViT paths are implemented. Set RUN_EXTRA_ARCHITECTURES=True to run them.')


In [ ]:
from pathlib import Path
import hashlib

HISTO_URL = (
    "https://github.com/ozanciga/self-supervised-histopathology/"
    "releases/download/tenpercent/tenpercent_resnet18.ckpt"
)

# SHA256 of the official release file checked when preparing this cell.
HISTO_SHA256 = (
    "61dc3ce7273f4ecb2960c3df31b67e36c38e4f30d9622b42496f842752992117"
)

HISTO_PATH = PROJECT_DIR / "weights" / "tenpercent_resnet18.ckpt"
HISTO_PATH.parent.mkdir(parents=True, exist_ok=True)

# Download once, then reuse the local file.
if not HISTO_PATH.exists():
    torch.hub.download_url_to_file(
        HISTO_URL, str(HISTO_PATH), progress=True
    )

with HISTO_PATH.open("rb") as f:
    digest = hashlib.file_digest(f, "sha256").hexdigest()

if digest != HISTO_SHA256:
    raise ValueError(
        "Checkpoint checksum mismatch. "
        "Remove the file and download it again."
    )

# The original checkpoint contains old PyTorch Lightning callback metadata.
# This inert placeholder allows restricted loading without installing Lightning.
class _LegacyCheckpointMetadata:
    pass

with torch.serialization.safe_globals([
    (
        _LegacyCheckpointMetadata,
        "pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint",
    )
]):
    original = torch.load(
        HISTO_PATH,
        map_location="cpu",
        weights_only=True,
    )

set_seed()

# Initialise the architecture, then load its histology-pretrained backbone.
histo_model = models.resnet18(weights=None)
histo_model.fc = nn.Identity()

# Remove the checkpoint prefix and exclude the SSL projection head.
backbone_state = {
    key.removeprefix("model.resnet."): value
    for key, value in original["state_dict"].items()
    if key.startswith("model.resnet.")
    and not key.startswith("model.resnet.fc.")
}

# Strict loading ensures that every backbone parameter is restored.
histo_model.load_state_dict(backbone_state, strict=True)

# New randomly initialised classifier for our five tissue categories.
histo_model.fc = nn.Linear(512, len(CLASSES))

del original, backbone_state

print("Histology-pretrained backbone loaded.")
print("New classifier:", histo_model.fc)

In [ ]:
name = "transfer_resnet18_histology"

if name in ARTIFACTS or (OUT / f"{name}.pt").exists():
    raise ValueError(
        "This experiment already exists. "
        "Use a fresh run directory to repeat it."
    )

start = time.perf_counter()

# Stage 1: train only the new classifier.
configure_transfer(
    histo_model,
    histo_model.fc,
    histo_model.layer4,
    stage="head",
)

best, first, checkpoint = fit_stage(
    model=histo_model,
    name=name,
    arch="resnet18",
    epochs=HEAD_EPOCHS,
    lr=1e-3,
    freeze_bn=True,
    stage="head",
)

# Start fine-tuning from the best head-training checkpoint.
histo_model.load_state_dict(
    load_checkpoint(checkpoint)["state_dict"]
)

# Stage 2: train the classifier and the last ResNet block.
configure_transfer(
    histo_model,
    histo_model.fc,
    histo_model.layer4,
    stage="last_block",
)

_, second, checkpoint = fit_stage(
    model=histo_model,
    name=name,
    arch="resnet18",
    epochs=FINETUNE_EPOCHS,
    lr=1e-5,
    freeze_bn=True,
    best_score=best,
    stage="last_block",
)

# Record the actual pretraining source.
source = {
    "type": "histology_ssl",
    "url": HISTO_URL,
    "sha256": digest,
}

saved = load_checkpoint(checkpoint)
saved["pretrained_initialisation"] = True
saved["pretraining_source"] = source
torch.save(saved, checkpoint)
del saved

# Reload the best checkpoint, compute validation metrics,
# save learning curves, and add this experiment to the comparison.
finish_neural(
    model=histo_model,
    name=name,
    arch="resnet18",
    checkpoint=checkpoint,
    history=first + second,
    seconds=time.perf_counter() - start,
)

ARTIFACTS[name]["pretraining_source"] = source

(OUT / f"{name}_source.json").write_text(
    json.dumps(source, indent=2),
    encoding="utf-8",
)

del histo_model
gc.collect()

display(
    pd.DataFrame(RESULTS).sort_values(
        "macro_f1", ascending=False
    )
)

### Extension: controlled architecture comparison
Run EfficientNet-B0 and ViT-B/16 by enabling `RUN_EXTRA_ARCHITECTURES` before executing the configuration cell, then run the notebook again. Use 224-pixel inputs and reduce BATCH_SIZE if necessary. They require additional downloads, memory and time; no fixed runtime is promised. Compare equal training budgets and report any changes. To test augmentation separately, rerun from the same seed and initial weights with only augmentation changed. Never compare runs with different data subsets without making the difference explicit.


## Task 10 — Combine handcrafted and deep features


Build a fresh ImageNet ResNet18 encoder with its classification head replaced by `Identity`. Freeze all parameters, use evaluation mode and deterministic transforms. Extract 512-dimensional embeddings in the **same row order** as the input DataFrame. No fine-tuning occurs in this experiment.

Concatenate the 70 handcrafted descriptors with the 512 deep features. Fit separate train-only StandardScaler/logistic-regression pipelines for deep-only and hybrid features; compare both with the existing handcrafted-only baseline. This isolates whether handcrafted information adds value to the frozen deep representation.

**Contracts:** `make_encoder(pretrained=True)` returns the frozen encoder. `extract_embeddings(encoder, frame)` returns float array (len(frame),512). `combine_features(H,D)` returns (N,582). Discuss scaling, feature redundancy, and why concatenation might not help.


In [ ]:
def make_encoder(pretrained=True):
    # TODO: load ResNet18, replace fc with Identity, freeze parameters and set eval mode.
    raise NotImplementedError('Complete Task 10 encoder')

def extract_embeddings(encoder,frame):
    # TODO: deterministic DataLoader with shuffle=False, inference_mode, no weight updates.
    # Preserve input row order; concatenate CPU numpy embeddings to shape (N,512).
    raise NotImplementedError('Complete Task 10 embeddings')

def combine_features(H,D):
    # TODO: verify matching row counts and concatenate on the feature axis.
    raise NotImplementedError('Complete Task 10 feature fusion')


In [ ]:
assert combine_features(np.zeros((2,70)),np.zeros((2,512))).shape==(2,582)


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
set_seed();encoder=make_encoder(PRETRAINED)
start=time.perf_counter()
D_train=extract_embeddings(encoder,train);D_val=extract_embeddings(encoder,val)
embedding_seconds=time.perf_counter()-start
assert D_train.shape==(len(train),512) and D_val.shape==(len(val),512)
encoder_path=OUT/'frozen_resnet18_encoder.pt'
torch.save({'state_dict':{k:v.detach().cpu().clone() for k,v in encoder.state_dict().items()},
            'classes':CLASSES,'arch':'resnet18_encoder','image_size':IMAGE_SIZE},encoder_path)
np.savez_compressed(OUT/'deep_features.npz',train=D_train,val=D_val)
for key,A,B in [('deep_linear',D_train,D_val),
                ('hybrid',combine_features(H_train,D_train),combine_features(H_val,D_val))]:
    start=time.perf_counter();linear=fit_linear(A,y_train)
    record(key,ordered_probabilities(linear,B),
           {'kind':key,'estimator':linear,'encoder_checkpoint':str(encoder_path)},
           time.perf_counter()-start)
    joblib.dump(linear,OUT/f'{key}.joblib')
encoder.cpu();del encoder;gc.collect()
# fit_seconds excludes shared feature extraction; the two extraction times are saved
# separately in configuration.json. Neural timings cover training and validation.


## Task 11 — Compare validation results and analyse errors


Rank experiments by validation macro F1, then balanced accuracy, then method name (stable tie-breaking). Generate a per-class report and confusion matrix for the selected model. Display up to six most confident validation mistakes with true/predicted names and probabilities. If there are no errors, say so instead of inventing examples.

**Contracts:** `choose_model(results)` returns one registered method name. `show_errors(frame, probabilities, n=6)` returns a DataFrame of the shown mistakes and displays a figure when needed. Discuss why prevalence shift can change accuracy, why softmax scores are not calibrated certainty, and whether the best method's advantage is convincing. Do not use test results to choose a model.


In [ ]:
def choose_model(results):
    # TODO: sort metrics with the specified tie-breaks and return the leading method name.
    raise NotImplementedError('Complete Task 11 selection')

def show_errors(frame,probabilities,n=6):
    # TODO: add predicted IDs/confidence, select mistakes and sort by confidence.
    # Display original RGB patches with labels; handle the no-errors case.
    raise NotImplementedError('Complete Task 11 error analysis')


In [ ]:
SELECTED=choose_model(RESULTS)
assert SELECTED in ARTIFACTS
comparison=pd.DataFrame(RESULTS).sort_values('macro_f1',ascending=False)
display(comparison);print('Selected using validation only:',SELECTED)
prob=VAL_PROBS[SELECTED];prediction=prob.argmax(1)
report=classification_report(y_val,prediction,labels=np.arange(5),target_names=CLASSES,
                             output_dict=True,zero_division=0)
display(pd.DataFrame(report).T)
pd.DataFrame(report).T.to_csv(OUT/'validation_classification_report.csv')
ConfusionMatrixDisplay.from_predictions(y_val,prediction,labels=np.arange(5),
    display_labels=CLASSES,cmap='Blues',xticks_rotation=45)
plt.title('Validation: '+SELECTED);plt.tight_layout()
plt.savefig(OUT/'validation_confusion.png',dpi=140);plt.show()
errors=show_errors(val,prob);errors.to_csv(OUT/'validation_errors.csv',index=False)


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
# Supplied reloadable predictor: evaluates exactly the method selected on validation.
def predict_registered(artifact,frame):
    kind=artifact['kind']
    if kind=='constant': return np.eye(5)[np.full(len(frame),artifact['label'])]
    if kind=='classical':
        features=extract_handcrafted(frame)
        X=np.concatenate(list(features.values()),axis=1) if artifact['feature']=='all' else features[artifact['feature']]
        return ordered_probabilities(artifact['estimator'],X)
    if kind=='neural':
        checkpoint=load_checkpoint(artifact['checkpoint'])
        if checkpoint['image_size']!=IMAGE_SIZE: raise ValueError('Restore the saved image_size before inference.')
        model=SmallCNN() if artifact['arch']=='small_cnn' else build_transfer(artifact['arch'],False)[0]
        model.load_state_dict(checkpoint['state_dict']);model.to(DEVICE)
        probability=predict_neural(model,make_loader(frame,eval_transform))
        model.cpu();del model;gc.collect()
        return probability
    if kind in {'deep_linear','hybrid'}:
        model=make_encoder(False)
        checkpoint=load_checkpoint(artifact['encoder_checkpoint'])
        if checkpoint['image_size']!=IMAGE_SIZE: raise ValueError('Restore the saved image_size before inference.')
        model.load_state_dict(checkpoint['state_dict'])
        deep=extract_embeddings(model,frame)
        model.cpu();del model;gc.collect()
        if kind=='hybrid':
            hand=extract_handcrafted(frame)
            deep=combine_features(np.concatenate(list(hand.values()),axis=1),deep)
        return ordered_probabilities(artifact['estimator'],deep)
    raise ValueError(kind)

# Check that saved checkpoints/pipelines reproduce stored validation predictions.
reloaded_probability=predict_registered(ARTIFACTS[SELECTED],val)
np.testing.assert_allclose(reloaded_probability,VAL_PROBS[SELECTED],rtol=1e-4,atol=1e-5)


## Task 12 — Export a final evaluation after freezing the protocol


Implement an export function for a labelled final-test set. It must compute accuracy, balanced accuracy and macro F1, write a per-class report and confusion matrix, and write one row per image including its root-relative path, true class, predicted class and five named probability columns. This is a **local assessment CSV**, not an asserted Kaggle submission format.

**Contract:** `export_assessment(frame, probabilities, prefix)` writes files under OUT and returns the metric dictionary. Final-test execution is supplied below, guarded by RUN_FINAL_TEST. Before enabling it, finish every model/hyperparameter decision using validation, record the selected method, then run the final section once. Do not retrain or change choices after seeing test scores.


In [ ]:
def export_assessment(frame,probabilities,prefix):
    # TODO: check shape/normalisation, obtain argmax predictions and call scores.
    # Write metrics JSON, per-class report CSV, confusion PNG and prediction CSV.
    # Preserve relative_path and class names; name probability columns prob_<class>.
    raise NotImplementedError('Complete Task 12')


In [ ]:
# Exercise export on validation first; test remains untouched.
exported=export_assessment(val,VAL_PROBS[SELECTED],'selected_validation')
assert all(key in exported for key in ['accuracy','balanced_accuracy','macro_f1'])


**Your explanation:** Write your reasoning and interpret the results you obtain.


In [ ]:
# Save the selection and experiment configuration BEFORE any final-test read.
from importlib.metadata import version
for package in ['numpy','pandas','pillow','scikit-learn','scikit-image','matplotlib','joblib']:
    VERSIONS[package]=version(package)
configuration={'classes':CLASSES,'class_to_idx':CLASS_TO_IDX,'seed':SEED,
    'image_size':IMAGE_SIZE,'feature_size':FEATURE_SIZE,'batch_size':BATCH_SIZE,
    'epochs':{'cnn':CNN_EPOCHS,'head':HEAD_EPOCHS,'fine_tune':FINETUNE_EPOCHS},
    'max_train_per_class':MAX_TRAIN_PER_CLASS,'pretrained':PRETRAINED,
    'extra_architectures':RUN_EXTRA_ARCHITECTURES,'synthetic_smoke':SMOKE,
    'selection_metric':'validation macro F1','selected_method':SELECTED,
    'data_root':str(ROOT),'versions':VERSIONS,
    'training_rows':len(train),'validation_rows':len(val),
    'feature_extraction_seconds':{'handcrafted_train_val':handcrafted_seconds,
                                  'deep_train_val':embedding_seconds}}
(OUT/'configuration.json').write_text(json.dumps(configuration,indent=2),encoding='utf-8')
# Keeps fitted sklearn pipelines together with paths to this run's neural checkpoints.
joblib.dump(ARTIFACTS,OUT/'artifact_registry.joblib')
np.savez_compressed(OUT/'validation_probabilities.npz',**VAL_PROBS)
print('Saved configuration and model registry:',OUT)

if RUN_FINAL_TEST:
    test=scan_split(ROOT,'test').reset_index(drop=True)
    # Hash the full original train/val splits, including unused training-subset rows.
    test_audit=audit_frames([test])
    full_audit=pd.concat([audit,test_audit],ignore_index=True)
    overlap=find_cross_split_duplicates(full_audit)
    full_audit.to_csv(OUT/'all_splits_audit.csv',index=False)
    if not overlap.empty:
        overlap.to_csv(OUT/'final_test_overlap.csv',index=False)
        raise ValueError('Cross-split overlap detected: do not report final-test scores.')
    test_prob=predict_registered(ARTIFACTS[SELECTED],test)
    final_metrics=export_assessment(test,test_prob,'final_test')
    print('Final test — fixed method:',SELECTED,final_metrics)
    display(test.class_name.value_counts().reindex(CLASSES).rename('test_count'))
else:
    print('Test not opened. Finish selection, then set RUN_FINAL_TEST=True in this session and run this cell.')




**Practical troubleshooting:** if data are not found, set DATA_ROOT to the directory containing `train` and `val`; do not point it to a class or to the ZIP. If multiple datasets are detected, select the root explicitly. If dependencies were just installed, restart the kernel. For memory errors reduce BATCH_SIZE and restart the run; optionally set a balanced MAX_TRAIN_PER_CLASS and disclose it. A complete run can be expensive: no synthetic result is a substitute for training on the actual data. If ImageNet weights cannot download, enable internet or arrange a populated PyTorch cache; PRETRAINED=False is only a separately labelled random-initialisation experiment.

**Loading results later:** the registry stores checkpoint paths from its original run. After moving the output folder, update those paths to the new folder. Load only checkpoints/joblib files you created or trust. Keep this notebook's definitions, CLASSES and preprocessing configuration available when reloading.

### Sources
- [Kaggle competition](https://www.kaggle.com/competitions/artificial-intelligence-for-histopathology/) and the supplied challenge description/screenshots establish the task and local archive layout. Exact image counts and performance are determined at execution.
- [scikit-image texture features](https://scikit-image.org/docs/stable/api/skimage.feature.html) and [region measurements](https://scikit-image.org/docs/stable/api/skimage.measure.html).
- Torchvision: [ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html), [EfficientNet-B0](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html), [ViT-B/16](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html).
- [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) and [classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html).
